# Experimento 1 - Baseline: YOLO11n pré-treinado (sem treino)

Projeto: **Detecção de Faces para Anonimização de Imagens** (conformidade com a LGPD) - disciplina de Introdução à Ciência de Dados e Aprendizado de Máquina (IFSC).

Este notebook avalia o modelo **YOLO11n pré-treinado no COCO**, sem nenhum treinamento adicional, no conjunto de teste do nosso subset do WIDER Face. O objetivo é estabelecer um **baseline**: os experimentos de transfer learning (notebooks 02 e 03) precisam superar este resultado para justificar o custo de treinamento.

> **Como usar no Google Colab:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **GPU (T4)**. Depois execute as células em ordem.

## 1. Instalação das dependências

O Colab já traz OpenCV, Matplotlib e Pandas pré-instalados; instalamos apenas a biblioteca `ultralytics` (YOLO11) e confirmamos a versão e a disponibilidade de GPU. A checagem é feita diretamente com `torch.cuda.is_available()` - o utilitário `ultralytics.checks()` dispara subprocessos de inspeção do ambiente que podem travar em algumas máquinas.

In [9]:
%pip install -q "ultralytics>=8.3.0"

import torch
import ultralytics

print('ultralytics', ultralytics.__version__)
print('GPU disponivel:', torch.cuda.is_available())

Note: you may need to restart the kernel to use updated packages.
ultralytics 8.4.65
GPU disponivel: True


## 2. Parâmetros

- `TRAIN_SIZE`, `VAL_FRACTION`, `TEST_SIZE` e `SEED` controlam o subset gerado - devem ser **os mesmos em todos os notebooks** para que os experimentos sejam comparáveis.
- `IMGSZ` é a resolução de entrada da rede (640 é o padrão da família YOLO).
- `EM_COLAB` detecta o ambiente automaticamente, permitindo rodar o mesmo notebook localmente.

In [10]:
import sys
from pathlib import Path

EM_COLAB = 'google.colab' in sys.modules

REPO_URL = 'https://github.com/ifsc-sj-projetos-ia/corrige-aqui'

if EM_COLAB:
    BASE_DIR = Path('/content/projeto-faces')
elif Path.cwd().name == 'notebooks':
    BASE_DIR = Path.cwd().parent
else:
    BASE_DIR = Path.cwd()

NOME_EXPERIMENTO = 'exp1_baseline_pretreinado'
TRAIN_SIZE = 1500
VAL_FRACTION = 0.2
TEST_SIZE = 300
SEED = 42
IMGSZ = 640
CONF_VISUALIZACAO = 0.25

## 3. Obtenção e preparação do dataset

No Colab, clonamos o repositório e executamos `src/prepare_dataset.py`, que:

1. Baixa o WIDER Face (~2 GB na primeira execução - leva alguns minutos);
2. Converte as anotações para o formato YOLO (`classe cx cy w h` normalizados);
3. Sorteia o subset com a `SEED` fixa e gera os splits de treino/validação/teste.

O conjunto de **teste** vem da partição de validação original do WIDER Face, que nunca é usada no treinamento - isso elimina o risco de vazamento de dados entre treino e teste.

In [11]:
if EM_COLAB and not BASE_DIR.exists():
    !git clone "{REPO_URL}" "{BASE_DIR}"

DATA_DIR = BASE_DIR / 'data'
SUBSET_DIR = DATA_DIR / 'wider_subset'
DATASET_YAML = SUBSET_DIR / 'dataset.yaml'
SCRIPT_PREPARO = BASE_DIR / 'src' / 'prepare_dataset.py'

if not DATASET_YAML.exists():
    !"{sys.executable}" "{SCRIPT_PREPARO}" --data-dir "{DATA_DIR}" --train-size {TRAIN_SIZE} --val-fraction {VAL_FRACTION} --test-size {TEST_SIZE} --seed {SEED}

## 4. O modelo pré-treinado

O YOLO11n (*n* de *nano*, ~2,6 milhões de parâmetros) foi pré-treinado no **COCO**, um dataset com 80 classes de objetos - entre elas `person`, mas **nenhuma classe de rosto**. Por isso, avaliamos as detecções da classe `person` (classe 0) contra as caixas de rosto do nosso dataset: é o mais próximo que o modelo consegue oferecer sem treino.

A expectativa é um desempenho **muito baixo**: a caixa de uma pessoa inteira raramente atinge IoU ≥ 0,5 com a caixa que delimita apenas o rosto. É exatamente essa lacuna que motiva o transfer learning dos próximos experimentos.

A célula abaixo carrega o modelo e lista as primeiras classes do COCO para evidenciar o problema.

In [12]:
from ultralytics import YOLO

modelo = YOLO('yolo11n.pt')
list(modelo.names.items())[:10]

[(0, 'person'),
 (1, 'bicycle'),
 (2, 'car'),
 (3, 'motorcycle'),
 (4, 'airplane'),
 (5, 'bus'),
 (6, 'train'),
 (7, 'truck'),
 (8, 'boat'),
 (9, 'traffic light')]

## 5. Avaliação no conjunto de teste

Métricas usadas em todo o projeto (calculadas pela Ultralytics):

- **Precision** - das detecções feitas, quantas correspondem a rostos reais (IoU ≥ 0,5). Precision baixa significa borrar regiões sem rosto - custo baixo no nosso cenário de anonimização.
- **Recall** - dos rostos reais, quantos foram detectados. Recall baixo significa **rostos sem anonimização**: o erro mais grave do ponto de vista da LGPD.
- **mAP@0,5** - área sob a curva precision–recall com limiar de IoU 0,5; resume o compromisso entre as duas métricas ao variar o limiar de confiança.
- **mAP@0,5:0,95** - média do mAP para limiares de IoU de 0,5 a 0,95; exige localização mais precisa das caixas.

Usamos `split='test'` para avaliar no conjunto de teste e `classes=[0]` para manter apenas as detecções de `person`. Como o modelo ainda tem as 80 classes do COCO, a tabela impressa mostrará o nome `person`. Os gráficos (curva precision–recall etc.) são salvos em `models/val_exp1_baseline_pretreinado/`.

In [13]:
resultados_teste = modelo.val(
    data=str(DATASET_YAML),
    split='test',
    imgsz=IMGSZ,
    classes=[0],
    plots=True,
    project=str(BASE_DIR / 'models'),
    name=f'val_{NOME_EXPERIMENTO}',
    exist_ok=True,
)

Ultralytics 8.4.65  Python-3.14.3 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 147.460.5 MB/s, size: 111.2 KB)
val: Scanning C:\Users\Davi\corrige-aqui\data\wider_subset\test\labels.cache... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 240.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.9it/s 4.9s0.1s
                   all        300       1993    0.00543     0.0522   0.000909   0.000109
                person        300       1993    0.00543     0.0522   0.000909   0.000109
Speed: 4.5ms preprocess, 4.1ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to C:\Users\Davi\corrige-aqui\models\val_exp1_baseline_pretreinado


## 6. Registro das métricas

Cada experimento grava uma linha em `results/metrics.csv`; re-execuções substituem a linha do mesmo experimento. Para o baseline, registramos `epochs = 0` (nenhum treino).


In [14]:
import pandas as pd
from datetime import datetime

CSV_METRICAS = BASE_DIR / 'results' / 'metrics.csv'

metricas = {
    'mAP50': round(float(resultados_teste.box.map50), 4),
    'mAP50_95': round(float(resultados_teste.box.map), 4),
    'precision': round(float(resultados_teste.box.mp), 4),
    'recall': round(float(resultados_teste.box.mr), 4),
    'epochs': 0,
}

def salvar_metricas(nome_experimento, metricas, caminho_csv):
    linha = {'experimento': nome_experimento, **metricas, 'data_hora': datetime.now().isoformat(timespec='seconds')}
    caminho_csv.parent.mkdir(parents=True, exist_ok=True)
    if caminho_csv.exists():
        tabela = pd.read_csv(caminho_csv)
        tabela = tabela[tabela['experimento'] != nome_experimento]
        tabela = pd.concat([tabela, pd.DataFrame([linha])], ignore_index=True)
    else:
        tabela = pd.DataFrame([linha])
    tabela.to_csv(caminho_csv, index=False)
    return tabela

salvar_metricas(NOME_EXPERIMENTO, metricas, CSV_METRICAS)

,experimento,mAP50,mAP50_95,precision,recall,epochs,data_hora
0,exp2_feature_extraction,0.7101,0.3987,0.8426,0.6312,20,2026-06-11T10:29:30
1,exp1_baseline_pretreinado,0.0009,0.0001,0.0054,0.0522,0,2026-06-11T10:51:54


## 7. Análise qualitativa

Visualizamos as detecções em algumas imagens de teste. Observe que o modelo desenha caixas de **corpo inteiro** (classe `person`), e não de rosto - a sobreposição com as caixas anotadas de face é pequena, o que explica o mAP baixo medido acima.

In [ ]:
%matplotlib inline
import cv2
import matplotlib.pyplot as plt

imagens_teste = sorted((SUBSET_DIR / 'test' / 'images').glob('*.jpg'))[:6]
predicoes = modelo.predict([str(p) for p in imagens_teste], conf=CONF_VISUALIZACAO, classes=[0], verbose=False)

figura, eixos = plt.subplots(2, 3, figsize=(16, 9))
for eixo, predicao in zip(eixos.flat, predicoes):
    eixo.imshow(cv2.cvtColor(predicao.plot(), cv2.COLOR_BGR2RGB))
    eixo.axis('off')
figura.tight_layout()
plt.show()

## 8. Download dos resultados (Colab)

O armazenamento do Colab é apagado ao fim da sessão.

In [16]:
if EM_COLAB:
    from google.colab import files
    files.download(str(CSV_METRICAS))

## Conclusões do experimento

Pontos para discutir no relatório:

1. Por que o mAP@0,5 é tão baixo mesmo com um modelo considerado forte? (incompatibilidade entre a classe `person` do COCO e a tarefa de localizar rostos)
2. O que esse resultado diz sobre a necessidade de dados anotados específicos para a tarefa?
3. Este experimento cumpre o requisito da proposta de ter *um baseline simples para comparação* - os experimentos 02 e 03 devem ser comparados contra esta linha do `metrics.csv`.